In [ ]:
!pip install opendatasets


In [ ]:
import opendatasets as od
od.download("https://www.kaggle.com/datasets/jehanbhathena/weather-dataset/data")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: sathyan
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/jehanbhathena/weather-dataset


100%|██████████| 587M/587M [00:03<00:00, 179MB/s] 


In [ ]:
import tensorflow as tf

# Check if TensorFlow is built with CUDA
print(f"TensorFlow Built with CUDA: {tf.test.is_built_with_cuda()}")

# Check for available GPU devices
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ CUDA Enabled GPU Detected: {len(gpus)}")
    for gpu in gpus:
        print(f"   - Device: {gpu.name}")
        # Optional: Print GPU details
        !nvidia-smi
else:
    print("❌ No GPU detected. Please ensure 'T4 GPU' is selected in Runtime settings.")

TensorFlow Built with CUDA: True
✅ CUDA Enabled GPU Detected: 1
   - Device: /physical_device:GPU:0
Thu Nov 27 04:58:59 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             11W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                       

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
# distinct preprocessing for EfficientNet
from tensorflow.keras.applications.efficientnet import preprocess_input

# --- 1. Define Paths & Parameters ---
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
DATA_DIR = '/content/weather-dataset/dataset' # Path to your dataset

# --- 2. Create Generators (Specific to EfficientNet) ---
# Note: EfficientNet's preprocess_input scales pixels to [-1, 1] or [0, 1] automatically
train_datagen_eff = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.20,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

val_datagen_eff = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.20
)

# --- 3. Load Images ---
print("\nLoading Data for EfficientNet...")
train_generator_eff = train_datagen_eff.flow_from_directory(
    DATA_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_generator_eff = val_datagen_eff.flow_from_directory(
    DATA_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)


Loading Data for EfficientNet...
Found 5493 images belonging to 11 classes.
Found 1369 images belonging to 11 classes.


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam

# --- 1. Build the Model ---
print("Building EfficientNetB0...")
# Load EfficientNetB0 pre-trained on ImageNet
# include_top=False removes the original 1000-class output
base_model_eff = EfficientNetB0(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze the base model initially
base_model_eff.trainable = False

# Add our custom classification head
x = base_model_eff.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x) # Helps prevent overfitting
predictions = Dense(11, activation='softmax')(x) # 11 weather classes

model_eff = Model(inputs=base_model_eff.input, outputs=predictions)

# --- 2. Compile & Train Baseline (Frozen) ---
model_eff.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\n--- Starting Phase 1: Baseline Training (Frozen Base) ---")
history_eff_baseline = model_eff.fit(
    train_generator_eff,
    epochs=10,
    validation_data=val_generator_eff
)

# --- 3. Fine-Tuning ---
print("\n--- Starting Phase 2: Fine-Tuning ---")

# Unfreeze the base model
base_model_eff.trainable = True

# Freeze the bottom layers
# EfficientNetB0 has roughly 230+ layers.
# We will freeze the first 200 and only fine-tune the very top.
for layer in base_model_eff.layers[:200]:
    layer.trainable = False

# Re-compile with a VERY LOW learning rate (Critical!)
model_eff.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_eff.summary()

history_eff_finetune = model_eff.fit(
    train_generator_eff,
    epochs=20, # Total epochs (starts at epoch 11)
    initial_epoch=10,
    validation_data=val_generator_eff
)

Building EfficientNetB0...
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

--- Starting Phase 1: Baseline Training (Frozen Base) ---


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 141s 656ms/step - accuracy: 0.5927 - loss: 1.2277 - val_accuracy: 0.8503 - val_loss: 0.4482
Epoch 2/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 77s 450ms/step - accuracy: 0.8336 - loss: 0.4940 - val_accuracy: 0.8729 - val_loss: 0.3818
Epoch 3/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 79s 457ms/step - accuracy: 0.8730 - loss: 0.3983 - val_accuracy: 0.8722 - val_loss: 0.3806
Epoch 4/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 77s 445ms/step - accuracy: 0.8726 - loss: 0.3641 - val_accuracy: 0.8685 - val_loss: 0.3588
Epoch 5/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 78s 455ms/step - accuracy: 0.8876 - loss: 0.3289 - val_accuracy: 0.8766 - val_loss: 0.3586
Epoch 6/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 77s 449ms/step - accuracy: 0.8936 - loss: 0.3110 - val_accuracy: 0.8751 - val_loss: 0.3542
Epoch 7/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 79s 456ms/step - accuracy: 0.9008 - loss: 0.2828 - val_accuracy: 0.8773 - val_loss: 0.3619
Epoch 8/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 78s 455ms/step - accuracy: 0.9027 - loss: 

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,214,958 (16.08 MB)

 Trainable params: 2,216,091 (8.45 MB)

 Non-trainable params: 1,998,867 (7.63 MB)

Epoch 11/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 135s 616ms/step - accuracy: 0.8142 - loss: 0.5664 - val_accuracy: 0.8641 - val_loss: 0.4153
Epoch 12/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 79s 459ms/step - accuracy: 0.8485 - loss: 0.4387 - val_accuracy: 0.8634 - val_loss: 0.4153
Epoch 13/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 79s 459ms/step - accuracy: 0.8693 - loss: 0.3772 - val_accuracy: 0.8656 - val_loss: 0.4032
Epoch 14/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 79s 460ms/step - accuracy: 0.8885 - loss: 0.3324 - val_accuracy: 0.8678 - val_loss: 0.3900
Epoch 15/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 78s 455ms/step - accuracy: 0.8927 - loss: 0.3227 - val_accuracy: 0.8766 - val_loss: 0.3772
Epoch 16/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 78s 454ms/step - accuracy: 0.9009 - loss: 0.2955 - val_accuracy: 0.8809 - val_loss: 0.3689
Epoch 17/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 79s 458ms/step - accuracy: 0.8994 - loss: 0.3037 - val_accuracy: 0.8824 - val_loss: 0.3624
Epoch 18/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 78s 453ms/step - accuracy: 0.9055 

In [ ]:
print("Evaluating EfficientNetB0...")
results_eff = model_eff.evaluate(val_generator_eff)
print(f"\n EfficientNet Test Loss: {results_eff[0]:.4f}")
print(f"EfficientNet Test Accuracy: {results_eff[1]:.4f}")

Evaluating EfficientNetB0...
43/43 ━━━━━━━━━━━━━━━━━━━━ 4s 98ms/step - accuracy: 0.8851 - loss: 0.3818

 EfficientNet Test Loss: 0.3491
EfficientNet Test Accuracy: 0.8897


Grad-CAM's Mechanism : It's a heatmap. It uses gradients (the learning signals) flowing back into the final convolutional layer to decide which parts of the image were "important" for the prediction.

In [ ]:
!pip install tf-keras-vis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.efficientnet import preprocess_input
from tf_keras_vis.gradcam import Gradcam
from tf_keras_vis.utils.model_modifiers import ReplaceToLinear
from tf_keras_vis.utils.scores import CategoricalScore

# --- 1. Setup ---
if 'model_eff' not in globals():
    raise NameError("❌ 'model_eff' is not defined. Please run EfficientNet training first.")

target_model = model_eff
LAST_CONV_LAYER = 'top_conv' # EfficientNet last conv layer

# Get Indices
if 'train_generator_eff' in globals():
    class_indices = train_generator_eff.class_indices
else:
    # Fallback
    class_indices = {'rain': 6, 'fogsmog': 1} # Verify these indices if generator is missing!

rain_index = class_indices['rain']
fogsmog_index = class_indices['fogsmog']

# Initialize Grad-CAM
gradcam = Gradcam(target_model,
                  model_modifier=ReplaceToLinear(),
                  clone=False)

# --- 2. Select Multiple Images ---
# You can manually list paths, or grab a random batch
# Here we grab a batch from the validation generator for variety
print("Selecting images...")
# Get a batch
# val_generator_eff yields (images, labels)
# The images are ALREADY preprocessed, so we need to be careful
(img_batch_preprocessed, label_batch) = next(val_generator_eff)

# Let's pick the first 5 images from this batch
NUM_IMAGES = 5
selected_images = img_batch_preprocessed[:NUM_IMAGES]
selected_labels = label_batch[:NUM_IMAGES]

# --- 3. Generate Heatmaps for the Batch ---
print(f"Generating heatmaps for {NUM_IMAGES} images...")

# A. Heatmaps for "Rain"
score_rain = CategoricalScore(rain_index)
cam_rain_batch = gradcam(score_rain,
                         selected_images,
                         penultimate_layer=LAST_CONV_LAYER)

# B. Heatmaps for "FogSmog"
score_fog = CategoricalScore(fogsmog_index)
cam_fog_batch = gradcam(score_fog,
                        selected_images,
                        penultimate_layer=LAST_CONV_LAYER)

# --- 4. Visualization Loop ---
# We need to 'deprocess' the images to show them to humans
# EfficientNet preprocess scales to roughly [-1, 1] or [0, 255] depending on version
# We'll normalize to [0, 1] for matplotlib
def deprocess_image(img):
    img = img - img.min()
    img = img / (img.max() - img.min())
    return img

plt.figure(figsize=(18, 4 * NUM_IMAGES))

for i in range(NUM_IMAGES):
    # Original Image (Restored for display)
    disp_img = deprocess_image(selected_images[i])

    # True Label
    true_label_idx = np.argmax(selected_labels[i])
    # Invert the dictionary to get name from index
    idx_to_class = {v: k for k, v in class_indices.items()}
    true_classname = idx_to_class.get(true_label_idx, "Unknown")

    # Plot Original
    ax = plt.subplot(NUM_IMAGES, 3, i * 3 + 1)
    ax.imshow(disp_img)
    ax.set_title(f"Image {i+1}: True Class = '{true_classname}'", fontsize=12)
    ax.axis('off')

    # Plot Rain Heatmap
    heatmap_r = np.uint8(plt.cm.jet(cam_rain_batch[i])[..., :3] * 255)
    ax = plt.subplot(NUM_IMAGES, 3, i * 3 + 2)
    ax.imshow(disp_img)
    ax.imshow(heatmap_r, cmap='jet', alpha=0.5)
    ax.set_title(f"Focus for 'Rain'", fontsize=12, color='green')
    ax.axis('off')

    # Plot Fog Heatmap
    heatmap_f = np.uint8(plt.cm.jet(cam_fog_batch[i])[..., :3] * 255)
    ax = plt.subplot(NUM_IMAGES, 3, i * 3 + 3)
    ax.imshow(disp_img)
    ax.imshow(heatmap_f, cmap='jet', alpha=0.5)
    ax.set_title(f"Focus for 'FogSmog'", fontsize=12, color='red')
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Manual Input of Your Results ---
# Replace these numbers with the exact values you got from your separate runs
vgg_acc = 74.90
resnet_acc = 89.77
effnet_acc = 89.19

# --- Data ---
models = ['VGG-19', 'EfficientNetB0', 'ResNet50']
accuracies = [vgg_acc, effnet_acc, resnet_acc]
colors = ['#FF9999', '#66B2FF', '#99FF99']

# --- Plotting ---
plt.figure(figsize=(10, 6))
bars = plt.bar(models, accuracies, color=colors, edgecolor='black')

plt.title('Final Accuracy Comparison', fontsize=16, fontweight='bold')
plt.ylabel('Validation Accuracy (%)', fontsize=14)
plt.ylim(0, 100)

# Add text labels
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, height + 1,
             f'{height:.2f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('final_accuracy_comparison.png', dpi=300)
plt.show()

In [ ]:
import numpy as np
from sklearn.utils import class_weight
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam

# --- 1. Calculate Class Weights ---
print("Calculating Class Weights...")
# We need the class labels for all training images
train_labels = train_generator_eff.classes

# Compute weights (balanced)
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

# Convert to dictionary format for Keras
class_weights_dict = dict(enumerate(class_weights))

print("\nClass Weights Dictionary:")
for idx, weight in class_weights_dict.items():
    class_name = list(train_generator_eff.class_indices.keys())[idx]
    print(f"  {class_name}: {weight:.2f}")

# --- 2. Build a Fresh Model ---
# We rebuild to ensure we aren't just continuing the old training
print("\nBuilding Fresh EfficientNetB0...")
base_model_eff = EfficientNetB0(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Start Frozen
base_model_eff.trainable = False

x = base_model_eff.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(11, activation='softmax')(x)

model_weighted = Model(inputs=base_model_eff.input, outputs=predictions)

model_weighted.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# --- 3. Train with Weights (Phase 1) ---
print("\n--- Starting Weighted Training (Phase 1: Frozen) ---")
history_weighted_1 = model_weighted.fit(
    train_generator_eff,
    epochs=10,
    validation_data=val_generator_eff,
    class_weight=class_weights_dict # <--- The Key Addition
)

# --- 4. Fine-Tuning with Weights (Phase 2) ---
print("\n--- Starting Weighted Fine-Tuning (Phase 2) ---")

base_model_eff.trainable = True
# Freeze bottom 200 layers
for layer in base_model_eff.layers[:200]:
    layer.trainable = False

model_weighted.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_weighted_2 = model_weighted.fit(
    train_generator_eff,
    epochs=20, # Total epochs (11-30)
    initial_epoch=10,
    validation_data=val_generator_eff,
    class_weight=class_weights_dict # <--- Still using weights
)

# --- 5. Final Evaluation ---
print("\nEvaluating Weighted Model...")
results = model_weighted.evaluate(val_generator_eff)
print(f"\n✅ Final Accuracy with Class Weights: {results[1]*100:.2f}%")